# Phase 1 : EDA et compréhension métier
## Moteur de tarification IARD, données freMTPL2

**Objectif** : connaître la donnée mieux que personne avant de modéliser.
Analyse de l'exposition, de la fréquence (surdispersion, excès de zéros),
de la sévérité (queue épaisse), cadre actuariel (offset, décomposition
fréquence x coût), qualité des données, feature engineering initial.

**Données** : freMTPL2freq (id 41214) + freMTPL2sev (id 41215) via OpenML,
jointure à gauche préservant les 678 013 polices. Portefeuille auto français.

## Journal des décisions et anomalies

Table des arbitrages de nettoyage et de modélisation, alimentée au fil de l'EDA.

| # | Constat | Décision | Justification | Impact |
|---|---------|----------|---------------|--------|
| 1 | Jointure : 678 013 lignes = 678 013 IDpol uniques, 0 NaN | Jointure validée | `ClaimAmount` agrégé en somme par police avant jointure | Une ligne = une police |
| 2 | 9 116 polices avec `ClaimNb > 0` mais `ClaimAmount = 0` (27 % des polices sinistrées) | Gardées pour la fréquence, exclues de la sévérité | Anomalie source de freMTPL2 : sinistres comptés sans montant dans la table sev. On ne modélise pas un coût non observé | Échantillon sévérité = 24 944 polices |
| 3 | `Exposure` > 1 pour 1 224 polices | Plafonné à 1 (appliqué) | Exposition annuelle bornée à 1 par construction | Fréquence +0.04 %, négligeable |
| 4 | `ClaimNb` jusqu'à 16 (9 polices > 4, exposition courte) | Écrêté à 4 (appliqué) | Aberrant sur contrat individuel ; littérature plafonne à 4 ; écrêtage préserve l'observation | Fréquence quasi inchangée |
| 5 | Sentinelles d'âge : `DrivAge` = 99 (70 polices), `VehAge` = 99/100 (48 polices) | Non modifiées ; neutralisées par bandes d'âge (section 7) | Pics isolés en rupture de distribution, profils quelconques : codes "inconnu" et non âges réels. Volume négligeable | Traité au feature engineering, sans imputation |
| 6 | Ratio variance/moyenne brut = 1.08 ; 95 % de zéros | Poisson retenu par défaut, pas de ZIP | Zéros conformes à Poisson (94.8 % prédits) ; surdispersion à trancher en Phase 2 sur dispersion résiduelle | Choix de loi différé Phase 2 |
| 7 | Gradient de fréquence monotone par zone (0.08 vers 0.14) | `Area` conservée comme variable explicative | Signal urbanisation cohérent ; proxy à auditer Phase 4 | Insight métier n°1 |
| 8 | 95 % de zéros dans `ClaimNb` | Pas de modèle zéro-inflaté | Excès de zéros = écart à $e^{-\lambda}$, pas quantité absolue. Ici 95.0 % observés vs 94.8 % prédits par Poisson : pas d'excès. Toutes les polices ont une exposition > 0, pas de zéros structurels | ZIP/hurdle écartés ; ordre Phase 2 : Poisson puis NB si surdispersion |
| 9 | Atomes de probabilité massifs sur le coût moyen (1 204, 1 128.12, 1 172, 602 EUR) | Documentés, conservés | Montants forfaitaires distincts et légitimes du jeu source (règlements standardisés ou valeurs imputées) | Trace attendue dans les résidus Phase 2 |
| 10 | Coût moyen divisé par un dénominateur faux (`ClaimNb` inclut des sinistres sans montant) | Ajout de `ClaimNbSev`, régénération du parquet | Impact **agrégé** majeur : coût moyen portefeuille 1 659 EUR (/ClaimNb) contre 2 265 EUR (/ClaimNbSev). Impact **individuel** marginal : 1 seule police de l'échantillon sévérité mélange sinistres chiffrés et non chiffrés | Corrige la décomposition PP et le poids du GLM Gamma |
| 11 | Décomposition PP : fréquence (ClaimNb) et coût moyen (ClaimNbSev) portent sur des comptages différents | Fréquence sur `ClaimNb`, sévérité sur `ClaimNbSev`, facteur correctif $N^{\text{sev}}/N = 0.732$ à la recombinaison | Un sinistre déclaré est un risque réel (fréquence) ; seul un montant observé est estimable (sévérité). Le facteur rétablit PP = 167.11 EUR | Architecture de recombinaison Phase 2 |
| 12 | Nature des 9 658 sinistres sans montant | Traités comme sans indemnisation ; PP = 167.11 EUR | Exposition plus courte (0.51 vs 0.69), malus plus faible (58 vs 65), absence homogène et non structurée : accident de collecte, pas coût réel | Niveau de prime Phase 2 fixé à 167 EUR |
| 13 | Taux d'absence de montant homogène par zone (0.22 à 0.33) | Facteur correctif global 0.732, sans variation par profil | Pas de concentration géographique de l'anomalie | Recombinaison GLM simplifiée Phase 2 |
| 14 | Coût moyen jusqu'à 1 EUR (46 polices < 10 EUR) | Conservé, pas de plancher | Volume négligeable ; le GLM Gamma gère les petites valeurs positives ; un plancher inventerait de la donnée | À revoir si résidus Gamma problématiques en Phase 2 |
| 15 | `DrivAge` : distribution en L (surrisque jeune, plateau ensuite), pas en U | 6 bandes : 18-23, 24-28, 29-45, 46-60, 61-75, 76+ | Bandes fines où l'information se concentre (jeunes), regroupement large sur le plateau ; pas de bande "âgés à risque" (absente des données) | Feature `DrivAgeBand` ; sentinelle 99 absorbée dans 76+ |
| 16 | `VehAge` : fréquence en décroissance monotone (surrisque neuf comportemental, baisse ensuite) | 6 bandes : 0-1, 2-5, 6-11, 12-15, 16-19, 20+ | Bande fine pour le pic neuf (0.16), regroupement du plateau et des âges élevés en suivant le gradient ; effet inverse attendu sur la sévérité | Feature `VehAgeBand` ; sentinelles 99-100 absorbées dans 20+ |
| 17 | `Density` très asymétrique (1 à 27 000, médiane 393) ; fréquence croît de 0.08 à 0.13 avec la densité | Transformation `log(Density)` créée | Log comprime l'échelle et linéarise l'effet ; relation régulière en échelle log confirmée empiriquement sur les déciles | Feature `log_Density` ; redondance avec `Area` à examiner (#18) |
| 18 | `Area` est une discrétisation directe de `log_Density` (bornes emboîtées, corrélation 0.971) | Conserver `log_Density`, écarter `Area` du GLM | Éviter la colinéarité ; relation log-linéaire (forme continue sans perte), parcimonie, interprétabilité | `Area` en réserve pour robustesse Phase 2 ; `Density` = proxy à auditer Phase 4 |
| 19 | `DrivAge` et `BonusMalus` corrélés à -0.48 (âgés = meilleur bonus) | Les deux conservés | Partagent l'information "expérience" sans redondance (loin du seuil 0.8) ; bonus-malus capte le comportement, l'âge des effets distincts | Coefficients à lire conjointement ; VIF à surveiller Phase 2 |
| 20 | `VehBrand` : 11 marques, la plus petite (B14) à 1 % de l'exposition | Conservée telle quelle, aucun regroupement | Toutes les marques exploitables ; regrouper une seule modalité isolée n'a pas de sens | Feature `VehBrand` inchangée |
| 21 | `Region` : 22 modalités, 8 régions sous 1 % de l'exposition (fréquences erratiques) | Régions < 1 % regroupées dans "Autres" (22 vers 14 modalités) | Coefficient GLM non fiable sous ~1 500 années-police ; "Autres" à 5 % expo, fréquence 0.11 proche de la moyenne | Feature `RegionGrouped` |

## 0. Setup et chargement

Imports, options d'affichage, résolution robuste de la racine du projet et
chargement du portefeuille. La racine est trouvée via le marqueur `.git`, ce
qui rend les chemins valides quel que soit le dossier d'exécution du notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

# Affichage : toutes les colonnes, nombres lisibles
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")


def get_project_root(marker: str = ".git") -> Path:
    """Remonte l'arborescence depuis le dossier courant jusqu'à trouver
    le dossier contenant le marqueur (.git). Rend les chemins robustes,
    peu importe d'où le notebook est lancé.
    """
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Racine du projet introuvable (marqueur : {marker})")


PROJECT_ROOT = get_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "fremtpl2.parquet"

df = pd.read_parquet(DATA_PATH)
print(f"Dimensions : {df.shape}")
df.head()

## 1. Contrôles d'intégrité

Avant toute analyse, on vérifie que la jointure n'a pas corrompu les données.
freMTPL2sev contient une ligne par sinistre : une jointure naïve multiplierait
les polices multi-sinistres. On contrôle donc : une ligne = une police
(`len == nunique(IDpol)`), cohérence entre nombre de sinistres et montants,
valeurs manquantes, et bornes des variables clés (exposition, ClaimNb).

**Résultat** : jointure propre (aucune duplication, aucun NaN). Anomalie
majeure détectée sur les 9 116 polices sinistrées sans montant (voir journal, #2),
qui sépare la population de modélisation fréquence de celle de la sévérité.

In [ ]:
# 1. La jointure a-t-elle dupliqué des lignes ?
# freMTPL2sev contient une ligne par sinistre : une jointure naive multiplie
# les lignes des polices multi-sinistres. On doit avoir 1 ligne = 1 police.
print("Lignes            :", len(df))
print("IDpol uniques     :", df["IDpol"].nunique())

# 2. Types et valeurs manquantes
df.info()
print("\nValeurs manquantes :")
print(df.isna().sum())

# 3. Vue d'ensemble descriptive
df.describe(include="all").T

In [ ]:
# EXPOSITION 
# Fraction d'annee pendant laquelle la police est observee. C'est LE denominateur
# de la frequence, et deviendra l'offset log(Exposure) dans le GLM Poisson.
# Elle doit vivre dans l'intervalle (0, 1].
print("=== Exposition ===")
print(df["Exposure"].describe())
print("Exposition > 1  :", (df["Exposure"] > 1).sum())
print("Exposition <= 0 :", (df["Exposure"] <= 0).sum())

# --- NOMBRE DE SINISTRES ---
# Exces de zeros (attendu) et eventuel plafonnement / valeurs aberrantes.
print("\n=== ClaimNb ===")
print(df["ClaimNb"].value_counts().sort_index())

# --- COHERENCE SINISTRES / MONTANTS ---
# La prime pure se decompose en frequence x cout moyen : cette decomposition
# n'a de sens que si nombre de sinistres et montants sont coherents.
print("\n=== Coherence ===")
print("Polices avec ClaimNb > 0     :", (df["ClaimNb"] > 0).sum())
print("Polices avec ClaimAmount > 0 :", (df["ClaimAmount"] > 0).sum())

incoherence_1 = df[(df["ClaimNb"] == 0) & (df["ClaimAmount"] > 0)]
incoherence_2 = df[(df["ClaimNb"] > 0) & (df["ClaimAmount"] == 0)]
print("ClaimNb = 0 mais montant > 0        :", len(incoherence_1))
print("ClaimNb > 0 mais montant nul/absent :", len(incoherence_2))

## 2. Analyse de l'exposition et fréquence de référence

L'exposition $e_i$ est la fraction d'année pendant laquelle la police $i$ est
observée. C'est le dénominateur de la fréquence et le futur offset du GLM.

**Principe actuariel.** La fréquence du portefeuille n'est pas la moyenne des
comptages, mais un taux par année-police :

$$
\hat{\lambda} = \frac{\sum_{i} N_i}{\sum_{i} e_i}
$$

où $N_i$ est le nombre de sinistres de la police $i$. La moyenne naïve
$\frac{1}{n}\sum_i N_i$ traiterait une police observée 2 mois comme une police
observée 12 mois, d'où sa sous-estimation.

Dans le GLM Poisson, on modélise l'espérance du comptage proportionnellement à
l'exposition :

$$
\mathbb{E}[N_i] = e_i \, \exp\!\left(\mathbf{x}_i^\top \boldsymbol{\beta}\right)
\quad\Longleftrightarrow\quad
\log \mathbb{E}[N_i] = \underbrace{\log e_i}_{\text{offset}} + \mathbf{x}_i^\top \boldsymbol{\beta}
$$

Le terme $\log e_i$ est l'**offset** : un coefficient fixé à 1, non estimé, qui
transforme le modèle de comptage en modèle de taux.

**Résultats.**
- Fréquence de référence : $\hat{\lambda} = 0.1007$ sinistre / année-police
  (10.1 pour 100), valeur canonique de freMTPL2.
- Moyenne naïve : $\frac{1}{n}\sum_i N_i = 0.0532$, sous-estimée de moitié car
  l'exposition moyenne vaut $\bar{e} = 0.53$ an.
- Distribution : pic massif à $e_i = 1$, forte proportion de contrats partiels.

In [ ]:
# --- VOLUME REEL DU PORTEFEUILLE ---
# La somme des expositions = nombre d'annees-polices observees.
# C'est le vrai "volume" d'assurance, pas le nombre de lignes.
total_expo = df["Exposure"].sum()
total_claims = df["ClaimNb"].sum()

print(f"Nombre de polices          : {len(df):,}")
print(f"Annees-polices (exposition): {total_expo:,.0f}")
print(f"Nombre total de sinistres  : {total_claims:,}")

# --- FREQUENCE DE REFERENCE ---
# Frequence = total sinistres / total exposition (un taux annuel).
freq_portefeuille = total_claims / total_expo
print(f"\nFrequence moyenne du portefeuille : {freq_portefeuille:.4f} sinistre / annee-police")
print(f"(soit environ {freq_portefeuille*100:.1f} sinistres pour 100 annees-police)")

# Comparaison avec la (mauvaise) moyenne naive.
print(f"\nMoyenne naive de ClaimNb (a NE PAS utiliser) : {df['ClaimNb'].mean():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution de l'exposition
sns.histplot(df["Exposure"], bins=50, ax=axes[0])
axes[0].axvline(1.0, color="red", linestyle="--", label="Exposition = 1 an")
axes[0].set_title("Distribution de l'exposition")
axes[0].set_xlabel("Exposition (annees)")
axes[0].legend()

# Zoom sur les expositions aberrantes (> 1)
sns.histplot(df.loc[df["Exposure"] > 1, "Exposure"], bins=30, ax=axes[1], color="orange")
axes[1].set_title(f"Zoom : expositions > 1 an ({(df['Exposure'] > 1).sum()} polices)")
axes[1].set_xlabel("Exposition (annees)")

plt.tight_layout()
plt.show()

**Lecture des distributions d'exposition.**

*Graphe de gauche (distribution complète).* Un pic dominant à $e_i = 1$
(environ 175 000 polices) rassemble les contrats présents toute l'année.
En dessous, un étalement continu de contrats partiels ($0 < e_i < 1$), avec
de légers regroupements aux valeurs rondes (notamment $e_i = 0.5$). C'est la
signature d'un portefeuille vivant : entrées (nouvelles souscriptions) et
sorties (résiliations, changements de véhicule) en cours d'année. Plus de la
moitié du portefeuille n'est donc pas observée sur l'année pleine
($\bar{e} = 0.53$), ce qui rend l'exposition non ignorable et justifie l'offset.

*Graphe de droite (zoom sur $e_i > 1$).* Les 1 224 expositions supérieures à 1
se concentrent juste au-dessus de 1 (majoritairement entre 1.0 et 1.15), avec
une queue clairsemée jusqu'à 2.01. Cohérent avec l'hypothèse de doublons de
contrats mal dédupliqués dans le jeu source, plutôt qu'un phénomène assurantiel
réel. Faible volume, concentré près de la borne : anomalie mineure (voir
impact chiffré ci-dessous).

In [ ]:
# Impact du plafonnement de l'exposition a 1 sur la frequence de reference.
expo_capee = df["Exposure"].clip(upper=1.0)
freq_capee = total_claims / expo_capee.sum()

print(f"Frequence sans plafonnement : {freq_portefeuille:.4f}")
print(f"Frequence avec plafonnement : {freq_capee:.4f}")
print(f"Ecart relatif               : {(freq_capee/freq_portefeuille - 1)*100:+.2f} %")

**Impact du plafonnement de l'exposition à 1.**

On mesure l'effet du plafonnement $e_i \mapsto \min(e_i, 1)$ sur la fréquence
de référence :

$$
\hat{\lambda}_{\text{brut}} = \frac{\sum_i N_i}{\sum_i e_i} = 0.1007
\qquad
\hat{\lambda}_{\text{plafonnée}} = \frac{\sum_i N_i}{\sum_i \min(e_i, 1)} = 0.1007
$$

L'écart relatif est de $+0.04\,\%$, négligeable. La raison est double : les
polices concernées sont peu nombreuses (1 224 sur 678 013, soit $0.18\,\%$) et
leur excès d'exposition au-dessus de 1 est faible (concentré entre 1.0 et 1.15).
La masse d'exposition retirée est donc marginale devant le total de
358 499 années-police.

**Décision.** On plafonnera à 1 en Phase 2, non pour corriger un biais
statistique (inexistant ici) mais pour la cohérence conceptuelle : une
exposition annuelle est bornée à 1 an par construction. La démarche est
"mesurer avant de décider" plutôt qu'un nettoyage appliqué par principe.

## 3. Distribution de la fréquence : zéros et surdispersion

**Excès de zéros ?** Sous une loi de Poisson $N \sim \mathcal{P}(\lambda)$ de
moyenne $\lambda = 0.0532$, la probabilité d'observer zéro sinistre est :

$$
\mathbb{P}(N = 0) = e^{-\lambda} = e^{-0.0532} \approx 0.948
$$

On observe 95.0 % de zéros. L'écart est négligeable : les zéros ne sont **pas**
en excès, ils sont exactement ce que Poisson prédit. Aucun modèle zéro-inflaté
(ZIP, hurdle) n'est justifié ; la faible fréquence suffit.

**Surdispersion ?** Le modèle de Poisson suppose l'équidispersion :

$$
\mathbb{V}(N) = \mathbb{E}(N) = \lambda
$$

Le ratio brut $\frac{\mathbb{V}(N)}{\mathbb{E}(N)} = 1.08$ n'est **pas** un
diagnostic valide. Sur des comptages à exposition hétérogène, on a en effet :

$$
\mathbb{V}(N) \approx \mathbb{E}(N)\left(1 + \lambda \, \frac{\mathbb{V}(e)}{\mathbb{E}(e)}\right) > \mathbb{E}(N)
$$

donc le ratio dépasse mécaniquement 1 même sous Poisson pur. Le vrai diagnostic
se fera en Phase 2 sur la dispersion de Pearson du GLM ajusté :

$$
\hat{\phi} = \frac{1}{n - p} \sum_{i} \frac{(N_i - \hat{N}_i)^2}{\hat{N}_i}
$$

Une binomiale négative ne sera envisagée que si $\hat{\phi} \gg 1$ après
intégration de l'offset et des covariables.

**Fréquence par zone.** Gradient monotone de $\hat{\lambda}_A = 0.08$ (rural)
à $\hat{\lambda}_F = 0.14$ (urbain dense). Signal actuariel fort : la fréquence
croît avec l'urbanisation. `Area` est aussi un proxy à auditer en Phase 4.

In [ ]:
# --- EXCES DE ZEROS ---
part_zeros = (df["ClaimNb"] == 0).mean()
print(f"Part de polices sans sinistre : {part_zeros:.1%}")

# --- SURDISPERSION (indicatif, sur comptages bruts) ---
# Attention : ce ratio n'est PAS un diagnostic valide sur exposition heterogene.
# Le vrai test se fera en Phase 2 sur la dispersion de Pearson du GLM ajuste.
moyenne_cnb = df["ClaimNb"].mean()
variance_cnb = df["ClaimNb"].var()
print(f"\nMoyenne de ClaimNb  : {moyenne_cnb:.4f}")
print(f"Variance de ClaimNb : {variance_cnb:.4f}")
print(f"Ratio variance/moyenne : {variance_cnb/moyenne_cnb:.2f}  (Poisson attend 1.0)")

# --- FREQUENCE PAR ZONE ---
# Frequence = somme(ClaimNb) / somme(Exposure) par groupe. JAMAIS la moyenne des ClaimNb.
freq_par_zone = (
    df.groupby("Area", observed=True)
      .apply(lambda g: pd.Series({
          "expo": g["Exposure"].sum(),
          "sinistres": g["ClaimNb"].sum(),
          "frequence": g["ClaimNb"].sum() / g["Exposure"].sum(),
          "n_polices": len(g),
      }), include_groups=False)
      .sort_index()
)
print("\nFrequence par zone (Area) :")
print(freq_par_zone)

In [ ]:
# Un modele Poisson simple (sans covariables) predit-il deja les zeros observes ?
# Si oui, les zeros ne sont PAS en exces : aucun modele zero-inflate necessaire.
lam = df["ClaimNb"].mean()
p0_poisson = stats.poisson.pmf(0, lam)      # P(N=0) sous Poisson
p0_observe = (df["ClaimNb"] == 0).mean()    # part reelle de zeros

print(f"P(0 sinistre) predite par Poisson : {p0_poisson:.4f}")
print(f"Part de zeros observee            : {p0_observe:.4f}")
print(f"Ecart                             : {(p0_observe - p0_poisson)*100:+.2f} points")

**Vérification de l'excès de zéros (résultat).**

$$
\mathbb{P}(N = 0) = e^{-\lambda} = 0.9481
\qquad
\widehat{\mathbb{P}}(N = 0)_{\text{observée}} = 0.9498
$$

Écart de $+0.16$ point. Un modèle de Poisson sans covariable prédit déjà la
quasi-totalité des zéros observés. Conclusion ferme : **pas d'excès de zéros**,
donc aucun modèle zéro-inflaté (ZIP, hurdle) n'est justifié. La faible fréquence
$\lambda \approx 0.053$ suffit à expliquer les 95 % de zéros. Le GLM Poisson
(avec offset) reste le socle approprié pour la fréquence.

## 4. Distribution de la sévérité : la queue épaisse

**Changement de population.** La sévérité se modélise conditionnellement à la
survenance d'un sinistre observé en montant, soit l'échantillon
$\mathcal{S} = \{i : N_i > 0 \text{ et } \text{ClaimAmount}_i > 0\}$
(24 944 polices), et non le portefeuille entier.

**Coût moyen par sinistre.** On travaille au niveau police, sur le montant
moyen par sinistre. Le dénominateur correct est $N_i^{\text{sev}}$
(`ClaimNbSev`), le nombre de sinistres **effectivement observés en montant**,
et non $N_i$ (`ClaimNb`, sinistres déclarés) qui inclut des sinistres sans
montant dans la table sev :

$$
C_i = \frac{\text{ClaimAmount}_i}{N_i^{\text{sev}}}, \qquad i \in \mathcal{S}
$$

Utiliser $N_i$ diviserait mécaniquement le coût moyen par un facteur faux pour
les 9 117 polices multi-sinistres partiellement renseignées (voir journal, #10).
$N_i^{\text{sev}}$ servira aussi de **poids** au GLM Gamma en Phase 2 : un coût
moyen calculé sur 3 sinistres est plus fiable que sur 1 seul.

**Ce qu'on cherche.** La distribution des coûts est notoirement asymétrique à
droite (queue épaisse) : une majorité de petits sinistres matériels, et quelques
sinistres corporels très coûteux. Cette asymétrie dicte le choix de la loi en
Phase 2 : Gamma ou log-normale plutôt que gaussienne. On quantifie l'asymétrie
(skewness), l'épaisseur de queue (kurtosis) et le poids des gros sinistres.

In [ ]:
# --- ECHANTILLON SEVERITE ---
# Polices avec sinistre ET montant observe.
sev = df[(df["ClaimNb"] > 0) & (df["ClaimAmount"] > 0)].copy()

# Cout moyen par sinistre : denominateur = ClaimNbSev (sinistres observes en montant).
sev["CoutMoyen"] = sev["ClaimAmount"] / sev["ClaimNbSev"]

print(f"Echantillon severite : {len(sev):,} polices")
print("\n=== Distribution du cout moyen par sinistre ===")
print(sev["CoutMoyen"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99, 0.999]))

# Asymetrie et epaisseur de queue.
print(f"\nSkewness (asymetrie) : {stats.skew(sev['CoutMoyen']):.1f}")
print(f"Kurtosis (queue)     : {stats.kurtosis(sev['CoutMoyen']):.1f}")
print("(loi normale : skewness = 0, kurtosis = 0)")

# --- POIDS DES GROS SINISTRES ---
seuil_99 = sev["CoutMoyen"].quantile(0.99)
charge_totale = sev["CoutMoyen"].sum()
charge_top1 = sev.loc[sev["CoutMoyen"] >= seuil_99, "CoutMoyen"].sum()
print(f"\nSeuil du 99e percentile : {seuil_99:,.0f} EUR")
print(f"Part de la charge portee par le top 1 % : {charge_top1/charge_totale:.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Echelle lineaire : illisible a cause de la queue, mais montre l'asymetrie brute.
sns.histplot(sev["CoutMoyen"], bins=100, ax=axes[0])
axes[0].set_title("Cout moyen par sinistre (echelle lineaire)")
axes[0].set_xlabel("Cout moyen (EUR)")

# Echelle log : revele la structure d'une distribution a queue epaisse.
sns.histplot(np.log10(sev["CoutMoyen"]), bins=100, ax=axes[1], color="teal")
axes[1].set_title("Cout moyen par sinistre (echelle log10)")
axes[1].set_xlabel("log10(cout moyen)")

plt.tight_layout()
plt.show()

**Lecture de la distribution du coût moyen.**

*Asymétrie extrême.* Skewness $= 116.6$ et kurtosis $= 15\,815$ (loi normale :
0 et 0). La moyenne (2 221 EUR) écrase la médiane (1 172 EUR), signature d'une
queue droite très épaisse. Maximum à 4 075 400 EUR : sinistre corporel grave.

*Concentration de la charge.* Le top 1 % des sinistres (au-delà de 16 327 EUR)
porte **37.1 %** de la charge totale. Quelques centaines de polices concentrent
plus du tiers du coût. C'est le fondement économique de la réassurance et la
raison pour laquelle la question de l'écrêtement des sinistres graves se pose
en tarification.

*Effet d'échelle.* Le graphe en échelle linéaire est illisible : le maximum à
4 M EUR écrase tout le portefeuille en une barre. En échelle $\log_{10}$, la
distribution devient grossièrement symétrique et unimodale autour de
$\log_{10}(C) \approx 3$ (soit $\approx 1\,000$ EUR). Cette symétrie en log est
l'indice visuel qui oriente vers une **log-normale** ou une **Gamma** (dont le
lien log linéarise la moyenne), et exclut la gaussienne.

*Anomalie : masses ponctuelles.* Médiane $= 1\,172$ EUR mais $Q_{75} = 1\,228$
EUR : un quart de l'échantillon est comprimé sur 56 EUR d'amplitude. Les
`value_counts` confirment des **atomes de probabilité** (1 204, 1 128.12, 1 172,
602 EUR répétés des milliers de fois) : montants forfaitaires du jeu source.
Aucune loi continue ne reproduit un atome : trace attendue dans les résidus
en Phase 2.

*Borne basse.* Minimum à 1 EUR, sans réalité économique. À inspecter section 6.

In [ ]:
# Les montants forfaitaires repetes : atomes de probabilite du jeu source.
print(sev["CoutMoyen"].value_counts().head(10))

**Contrôle de la correction du dénominateur.**

La correction n'affecte qu'**une seule police** dans l'échantillon sévérité :
les sinistres sans montant appartiennent presque tous à des polices entièrement
non chiffrées (`ClaimAmount = 0`), déjà exclues de l'échantillon. L'effet réel
de la correction se situe donc au **niveau agrégé** (décomposition de la prime
pure, section 5), pas sur la distribution individuelle du coût moyen.

In [ ]:
# Polices ou le denominateur differe : la correction change leur cout moyen.
ecart = df[(df["ClaimNb"] > df["ClaimNbSev"]) & (df["ClaimAmount"] > 0)].copy()
ecart["CoutMoyen_faux"] = ecart["ClaimAmount"] / ecart["ClaimNb"]
ecart["CoutMoyen_correct"] = ecart["ClaimAmount"] / ecart["ClaimNbSev"]

print(f"Polices concernees (ClaimNb > ClaimNbSev ET montant > 0) : {len(ecart):,}")
print(ecart[["IDpol", "ClaimNb", "ClaimNbSev", "ClaimAmount",
             "CoutMoyen_faux", "CoutMoyen_correct"]].head(8))

## 5. Décomposition de la prime pure

La **prime pure** est le coût moyen annuel du risque par police, la cible
ultime de la tarification. On ne la modélise pas directement mais on la
décompose en fréquence x coût moyen, car ces deux composantes répondent à
des facteurs de risque distincts (un profil peut sinistrer souvent sans
sinistres coûteux, ou l'inverse).

**Prime pure de référence du portefeuille** (vérité économique) :

$$
PP = \frac{\sum_i \text{ClaimAmount}_i}{\sum_i e_i}
$$

**Décomposition.** Le télescopage n'est exact que si le *même* comptage de
sinistres $N$ apparaît dans les deux facteurs :

$$
PP = \underbrace{\frac{\sum_i N_i}{\sum_i e_i}}_{\text{fréquence}}
     \times
     \underbrace{\frac{\sum_i \text{ClaimAmount}_i}{\sum_i N_i}}_{\text{coût moyen}}
$$

C'est précisément là que l'anomalie du dénominateur (journal #10) refait
surface : la fréquence naturelle se calcule sur $N_i$ (`ClaimNb`, sinistres
déclarés) alors que le coût moyen correct se calcule sur $N_i^{\text{sev}}$
(`ClaimNbSev`, sinistres observés en montant). Mélanger les deux comptages
casse la cohérence. On le quantifie ci-dessous.

In [ ]:
# --- PRIME PURE DE REFERENCE DU PORTEFEUILLE ---
# Prime pure = charge totale / exposition totale (EUR par annee-police).
# C'est la verite economique incontestable : ni comptage de sinistres,
# ni denominateur discutable, juste la charge rapportee a l'exposition.
charge_totale = df["ClaimAmount"].sum()
total_expo = df["Exposure"].sum()
prime_pure = charge_totale / total_expo

print(f"Charge totale        : {charge_totale:,.0f} EUR")
print(f"Exposition totale    : {total_expo:,.0f} annees-police")
print(f"Prime pure reference : {prime_pure:,.2f} EUR / annee-police")

In [ ]:
# --- DECOMPOSITION FREQUENCE x SEVERITE ---
# Le telescopage PP = frequence x cout_moyen n'est EXACT que si le meme
# comptage de sinistres apparait dans les deux facteurs. On compare les
# deux conventions : sinistres declares (ClaimNb) vs observes en montant (ClaimNbSev).
n_declares = df["ClaimNb"].sum()       # sinistres declares (table freq)
n_observes = df["ClaimNbSev"].sum()    # sinistres observes en montant (table sev)

freq_declaree = n_declares / total_expo
freq_observee = n_observes / total_expo

cout_moyen_declare = charge_totale / n_declares    # denominateur ClaimNb (biaise)
cout_moyen_observe = charge_totale / n_observes    # denominateur ClaimNbSev (correct)

print(f"Sinistres declares (ClaimNb)     : {n_declares:,}")
print(f"Sinistres observes (ClaimNbSev)  : {n_observes:,}")
print(f"Ecart (sinistres sans montant)   : {n_declares - n_observes:,}")
print()
print(f"Frequence declaree (ClaimNb)     : {freq_declaree:.4f}")
print(f"Frequence observee (ClaimNbSev)  : {freq_observee:.4f}")
print()
print(f"Cout moyen (/ClaimNb)            : {cout_moyen_declare:,.2f} EUR")
print(f"Cout moyen (/ClaimNbSev)         : {cout_moyen_observe:,.2f} EUR")
print()
# Chaque decomposition est coherente EN INTERNE (meme comptage des deux cotes).
print("=== Coherence du telescopage ===")
print(f"freq_declaree x cout_moyen(/ClaimNb)    : {freq_declaree * cout_moyen_declare:,.2f} EUR")
print(f"freq_observee x cout_moyen(/ClaimNbSev) : {freq_observee * cout_moyen_observe:,.2f} EUR")
print(f"Prime pure de reference                 : {prime_pure:,.2f} EUR")

In [ ]:
# --- RECONCILIATION DES DEUX COMPTAGES ---
#frequence sur ClaimNb (tout sinistre reel compte),
# cout moyen sur ClaimNbSev (seuls les montants observes sont fiables),
# corrige par le taux d'observation des montants.
taux_observation = n_observes / n_declares

pp_naif = freq_declaree * cout_moyen_observe          # incoherent (surestime)
pp_corrige = freq_declaree * cout_moyen_observe * taux_observation

print(f"Taux d'observation des montants : {taux_observation:.4f}")
print(f"PP sans correction (biaisee)    : {pp_naif:,.2f} EUR")
print(f"PP avec facteur correctif       : {pp_corrige:,.2f} EUR")
print(f"Prime pure de reference         : {prime_pure:,.2f} EUR")

**Lecture de la décomposition.**

Prime pure de référence : **167.11 EUR / année-police** (charge de 59.9 M EUR
sur 358 499 années-police). C'est la vérité économique du portefeuille.

Les deux conventions de comptage télescopent exactement vers 167.11 EUR, mais
répartissent le risque différemment :

| Convention | Fréquence | Coût moyen |
|------------|-----------|------------|
| ClaimNb (déclarés) | 0.1007 | 1 659 EUR |
| ClaimNbSev (observés) | 0.0738 | 2 266 EUR |

L'écart vient des 9 658 sinistres déclarés sans montant.

**Choix de modélisation (Phase 2) et hypothèse sous-jacente.** Fréquence sur
`ClaimNb`, coût moyen sur `ClaimNbSev`. La recombinaison des deux GLM dépend de
la nature des 9 658 sinistres déclarés sans montant :

- s'ils sont de **faux sinistres** (erreurs, sans suite, coût nul), la charge
  observée est complète, $PP = 167.11$ EUR, et un facteur correctif
  $N^{\text{sev}}/N = 0.732$ s'applique à la recombinaison. Ce choix est
  numériquement équivalent à modéliser le coût moyen sur `ClaimNb` directement ;
- s'ils sont de **vrais sinistres** au montant manquant, l'estimateur correct
  leur impute le coût moyen observé, soit $PP = 0.1007 \times 2\,266 = 228$ EUR
  sans facteur, et $167.11$ sous-estime alors la charge de 27 %.

Le facteur n'est donc pas une simple mise en cohérence algébrique mais une
décision tarifaire. Tranché en section 6 après inspection de la nature de ces
sinistres.

## 6. Qualité des données et valeurs aberrantes

On tranche les anomalies laissées en suspens (journal #3, #4, #5, #12). Méthode
constante : mesurer l'ampleur, comprendre la cause, décider, documenter l'impact
chiffré. Aucune coupe appliquée par principe.

Ordre de traitement :
1. Nature des 9 658 sinistres déclarés sans montant (tranche le niveau de prime pure)
2. Exposition > 1 (plafonnement)
3. `ClaimNb` extrêmes (plafonnement éventuel)
4. `VehAge` et `DrivAge` à 100 (valeurs sentinelles ?)
5. Borne basse des montants (1 EUR)

### 6.1 Nature des sinistres déclarés sans montant

Question (journal #12) : les 9 658 sinistres comptés dans `ClaimNb` mais absents
de la table sev sont-ils de vrais sinistres au montant manquant, ou de faux
sinistres (sans suite, erreurs) ? La réponse fixe la prime pure à 167 ou 228 EUR.

Faute d'information directe, on compare le **profil** des polices sinistrées sans
montant à celui des polices sinistrées avec montant. Des profils semblables
plaident pour de vrais sinistres (montant manquant par accident de collecte) ;
un profil distinct (ex. exposition anormalement courte) plaide pour des sinistres
sans suite.

In [ ]:
# --- PROFIL DES POLICES SINISTREES SELON PRESENCE DU MONTANT ---
# On isole les polices ayant declare au moins un sinistre.
sinistrees = df[df["ClaimNb"] > 0].copy()

# Deux groupes : montant present (au moins un sinistre chiffre) vs totalement absent.
sinistrees["a_montant"] = sinistrees["ClaimAmount"] > 0

print("Repartition des polices sinistrees :")
print(sinistrees["a_montant"].value_counts())
print(f"\nPart sans aucun montant : {(~sinistrees['a_montant']).mean():.1%}")

# Comparaison de profil entre les deux groupes sur les variables cles.
profil = sinistrees.groupby("a_montant").agg(
    n_polices=("IDpol", "size"),
    expo_moyenne=("Exposure", "mean"),
    claimnb_moyen=("ClaimNb", "mean"),
    drivage_median=("DrivAge", "median"),
    bonusmalus_moyen=("BonusMalus", "mean"),
    vehpower_moyen=("VehPower", "mean"),
)
print("\nProfil compare (False = sans montant, True = avec montant) :")
print(profil.T)

In [ ]:
# --- REPARTITION PAR ZONE ET NOMBRE DE SINISTRES DECLARES ---
# Si les sinistres sans montant se concentrent sur certains profils, le facteur
# correctif global (0.732) serait inadapte : il devrait varier par profil.
print("Taux de polices SANS montant par zone (Area) :")
tab = sinistrees.groupby("Area", observed=True)["a_montant"].agg(
    n="size", part_sans_montant=lambda s: (~s).mean()
)
print(tab)

# Le nombre de sinistres declares change-t-il la probabilite d'avoir un montant ?
print("\nPart avec montant selon ClaimNb declare :")
print(sinistrees.groupby("ClaimNb")["a_montant"].mean())

**Décision : les sinistres sans montant sont traités comme sans indemnisation.**

Faisceau d'indices convergents sur les 9 116 polices sinistrées sans montant :
- exposition moyenne plus courte (0.51 contre 0.69 an) : contrats résiliés ou
  entrés en cours d'année, dossiers non menés à terme ;
- `BonusMalus` plus faible (58 contre 65) : un sinistre réellement indemnisé
  alourdit le coefficient, un sinistre sans suite beaucoup moins ;
- taux d'absence de montant homogène par zone (0.22 à 0.33, sans structure
  géographique) : pas de profil de risque concentrant l'anomalie ;
- indépendance vis-à-vis de `ClaimNb` (taux stable à 0.73) : absence de montant
  quasi aléatoire, cohérente avec un accident de collecte, non structurelle.

**Conclusion.** Lecture retenue : sinistres majoritairement sans indemnisation
effective. Prime pure de référence **167.11 EUR**, facteur correctif global
$N^{\text{sev}}/N = 0.732$ validé (l'homogénéité par zone autorise un facteur
unique, sans variation par profil). Réserve assumée : interprétation la plus
probable, non prouvée ; l'alternative (imputation) donnerait 228 EUR.

### 6.2 Exposition supérieure à 1 an

Rappel (journal #3) : 1 224 polices ont une exposition > 1, jusqu'à 2.01, ce qui
est impossible pour une police annuelle. On a déjà mesuré l'impact sur la
fréquence de référence (+0.04 %, négligeable). On applique ici le plafonnement à 1
et on fige la décision.

In [ ]:
# --- PLAFONNEMENT DE L'EXPOSITION A 1 ---
# L'exposition annuelle est bornee a 1 par construction. Les valeurs > 1 sont
# des artefacts (doublons de contrats). Impact deja mesure comme negligeable.
n_avant = (df["Exposure"] > 1).sum()
df["Exposure"] = df["Exposure"].clip(upper=1.0)
n_apres = (df["Exposure"] > 1).sum()

print(f"Polices plafonnees : {n_avant}")
print(f"Exposition > 1 apres plafonnement : {n_apres}")
print(f"Exposition max apres : {df['Exposure'].max():.4f}")

### 6.3 Nombre de sinistres extrêmes

Rappel (journal #4) : `ClaimNb` monte jusqu'à 16, avec une poignée de polices
au-delà de 4. Sur un contrat auto individuel, plus de 4 sinistres dans l'année
est hautement improbable. La littérature de référence sur freMTPL2 (Noll,
Salzmann, Wüthrich 2018) plafonne à 4. On mesure d'abord le volume concerné,
puis on décide.

In [ ]:
# --- INSPECTION DES CLAIMNB ELEVES ---
# Combien de polices, quelle exposition, pour justifier ou non un plafonnement.
extremes = df[df["ClaimNb"] > 4]
print(f"Polices avec ClaimNb > 4 : {len(extremes)}")
print(f"Soit {len(extremes)/len(df)*100:.4f} % du portefeuille")
print(f"\nExposition de ces polices :")
print(extremes["Exposure"].describe())
print(f"\nDetail par ClaimNb :")
print(df[df["ClaimNb"] > 4]["ClaimNb"].value_counts().sort_index())

In [ ]:
# --- PLAFONNEMENT DE CLAIMNB A 4 ---
# 9 polices depassent 4 sinistres, toutes a exposition courte (mediane 0.33 an) :
# aberrant sur contrat auto individuel. On ECRETE a 4 (on ne supprime pas :
# l'exposition et les covariables restent valides). Aligne sur Noll, Salzmann,
# Wuthrich (2018).
n_ecretees = (df["ClaimNb"] > 4).sum()
df["ClaimNb"] = df["ClaimNb"].clip(upper=4)
print(f"Polices ecretees : {n_ecretees}")
print(f"ClaimNb max apres : {df['ClaimNb'].max()}")

# Verification : la frequence de reference bouge-t-elle ? (doit etre negligeable)
freq_apres = df["ClaimNb"].sum() / df["Exposure"].sum()
print(f"Frequence de reference apres ecretage : {freq_apres:.4f}")

**Décision : écrêtage de `ClaimNb` à 4.**

9 polices (0.0013 % du portefeuille) déclarent plus de 4 sinistres, jusqu'à 16,
toutes à exposition courte (médiane 0.33 an). Aberrant sur un contrat auto
individuel : erreurs de saisie ou contrats mal catégorisés (flotte/pro). On
écrête à 4 plutôt que de supprimer, pour conserver l'exposition et les
covariables valides. Aligné sur Noll, Salzmann, Wüthrich (2018). Impact sur la
fréquence de référence négligeable (volume infime). L'écrêtage ne touche que la
cible fréquence ; la sévérité, calculée sur `ClaimNbSev`, reste inchangée.

### 6.4 Âges extrêmes (VehAge et DrivAge)

Rappel (journal #5) : `VehAge` et `DrivAge` atteignent 100. Un conducteur
centenaire est improbable mais possible ; un véhicule de 100 ans dans un
portefeuille auto de masse est presque sûrement une valeur sentinelle (code par
défaut). On inspecte la concentration aux valeurs suspectes avant de décider,
car une valeur sentinelle produit un pic net (beaucoup de polices à exactement
la même valeur ronde), là où un âge réel s'étale.

In [ ]:
# --- INSPECTION DES AGES EXTREMES ---
# Une valeur sentinelle se trahit par un pic : beaucoup de polices exactement
# a la meme valeur ronde, en rupture avec la distribution alentour.
print("=== DrivAge : haut de la distribution ===")
print(df[df["DrivAge"] >= 90]["DrivAge"].value_counts().sort_index())

print("\n=== VehAge : haut de la distribution ===")
print(df[df["VehAge"] >= 90]["VehAge"].value_counts().sort_index())

# Zoom sur les valeurs pile a 100.
print(f"\nDrivAge == 100 : {(df['DrivAge'] == 100).sum()} polices")
print(f"VehAge == 100  : {(df['VehAge'] == 100).sum()} polices")

# Pour VehAge = 100 : ces vehicules ont-ils un profil coherent ?
# Un vrai vehicule de collection aurait une faible puissance/densite urbaine variable ;
# une sentinelle aura des profils quelconques disperses.
if (df["VehAge"] == 100).sum() > 0:
    print("\nProfil des vehicules a VehAge = 100 :")
    print(df[df["VehAge"] == 100][["VehPower", "VehBrand", "DrivAge", "Exposure"]].describe(include="all").T)

**Lecture des âges extrêmes.**

*DrivAge.* Décroissance régulière de 90 à 100 ans (167, 121, 66... jusqu'à 3
centenaires) : âges réels de conducteurs très âgés, pas d'aberration. **Mais**
un pic isolé à 99 ans (70 polices, en rupture avec les 5 de 98 ans) trahit une
valeur sentinelle probable (99 = code "âge inconnu"). Volume infime.

*VehAge.* Pic net à 99 (23) et 100 ans (25), en rupture totale avec la
distribution (médiane 6 ans). Le profil de ces véhicules est quelconque
(VehPower, marque, âge conducteur dispersés) : ce ne sont pas des véhicules de
collection mais des **valeurs sentinelles** (99/100 = "âge inconnu ou hors
barème"). 48 polices.

**Décision.** Aucune modification des données brutes. Les deux anomalies (volume
négligeable) seront neutralisées par le regroupement en bandes d'âge de la
section 7 : une bande haute (ex. "conducteur 75+", "véhicule 20+ ans") absorbe
ces valeurs sans imputation arbitraire. On documente, on agit au bon endroit.

### 6.5 Borne basse des montants de sinistres

Rappel : le coût moyen descend à 1 EUR, sans réalité économique (aucun sinistre
auto ne coûte 1 EUR). On inspecte les très petits montants pour décider d'un
éventuel seuil plancher. Ces valeurs peuvent être des franchises, des
enregistrements partiels ou des erreurs.

In [ ]:
# --- INSPECTION DES PETITS MONTANTS ---
# Reconstruire l'echantillon severite (le df a ete modifie par les plafonnements).
sev = df[(df["ClaimNb"] > 0) & (df["ClaimAmount"] > 0)].copy()
sev["CoutMoyen"] = sev["ClaimAmount"] / sev["ClaimNbSev"]

# Combien de sinistres sous des seuils economiquement douteux ?
for seuil in [1, 10, 50, 100]:
    n = (sev["CoutMoyen"] < seuil).sum()
    print(f"Cout moyen < {seuil:>4} EUR : {n:>5} polices ({n/len(sev)*100:.2f} %)")

print("\nLes 15 plus petits couts moyens :")
print(sev["CoutMoyen"].nsmallest(15).values)

## 7. Feature engineering initial

Préparation des variables pour la modélisation GLM (Phase 2). Un GLM est linéaire
dans l'échelle du lien : il faut donc rendre linéarisables les effets qui ne le
sont pas (âges en U, densité logarithmique) et regrouper les modalités trop rares
pour être estimées de façon fiable.

Transformations prévues :
1. Bandes d'âge conducteur (`DrivAge`) : effet en U, non linéaire
2. Bandes d'âge véhicule (`VehAge`) : effet non linéaire + neutralise les sentinelles
3. Transformation log de `Density` : distribution très étalée
4. Regroupement des modalités rares (`VehBrand`, `Region`)
5. Examen de la redondance `Area` / `Density`

Principe : on ne modifie pas les variables d'origine, on crée de nouvelles
colonnes. La donnée brute reste traçable.

### 7.1 Bandes d'âge conducteur

La fréquence en fonction de l'âge suit typiquement une courbe en U en assurance
auto : jeunes conducteurs (inexpérience) et conducteurs âgés (réflexes) plus
risqués, creux au milieu. Un GLM linéaire ne peut pas capter un U avec une
variable continue : on découpe en bandes, chaque tranche portant son propre
niveau de risque. On vérifie d'abord le U empiriquement, puis on cale les bornes.

In [ ]:
# --- VERIFICATION EMPIRIQUE DU U : frequence par tranche d'age fine ---
# Frequence = somme(ClaimNb) / somme(Exposure) par tranche (jamais la moyenne).
tranches_fines = pd.cut(df["DrivAge"], bins=range(18, 101, 5))
freq_age = (
    df.groupby(tranches_fines, observed=True)
      .apply(lambda g: pd.Series({
          "expo": g["Exposure"].sum(),
          "frequence": g["ClaimNb"].sum() / g["Exposure"].sum(),
          "n_polices": len(g),
      }), include_groups=False)
)
print(freq_age)

# Visualisation du U
fig, ax = plt.subplots(figsize=(10, 5))
freq_age["frequence"].plot(kind="bar", ax=ax, color="steelblue")
ax.axhline(df["ClaimNb"].sum() / df["Exposure"].sum(), color="red",
           linestyle="--", label="Frequence moyenne portefeuille")
ax.set_title("Frequence par tranche d'age conducteur (verification du U)")
ax.set_ylabel("Frequence (sinistres / annee-police)")
ax.set_xlabel("Tranche d'age")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Lecture de la fréquence par âge : un L, pas un U.**

Le signal le plus fort du portefeuille est le surrisque des jeunes conducteurs :
fréquence de **0.20 pour les 18-23 ans** (le double de la moyenne de 0.10), puis
**0.12 pour les 24-28 ans**. Au-delà, la fréquence chute et se stabilise sur un
**plateau autour de 0.09-0.11** jusqu'aux âges élevés.

Contrairement au schéma théorique en U attendu en assurance auto, **la branche
droite (surrisque des conducteurs âgés) est absente** de ce portefeuille : de 60
à 90 ans, la fréquence reste plate, sans remontée nette. Les variations visibles
en haut de distribution (0.11 sur les 83-88 ans, 0.06 sur les 93-98 ans) reposent
sur des effectifs trop faibles (1 777 et 86 polices) pour être significatives :
c'est du bruit, pas un signal.

**Conséquence pour le découpage.** La distribution est un L : fort décrochage
jeune, puis plateau. On cale donc des bandes fines en bas (18-23 et 24-28, où se
concentre l'information) et un regroupement large sur le plateau. Aucune bande
"âgés à risque" n'est créée, car les données ne la justifient pas. La bande haute
(76+), volontairement large, reste statistiquement robuste et absorbe la sentinelle
à 99 ans (section 6.4) sans imputation.

In [ ]:
# --- BANDES D'AGE CONDUCTEUR ---
# Calees sur le signal observe : un "L" (surrisque jeune marque, puis plateau),
# et non un "U" theorique. Les deux premieres bandes sont fines car c'est la que
# se concentre l'information. Le plateau est regroupe largement. La bande haute
# absorbe la sentinelle a 99 ans (voir section 6.4) sans imputation.
bornes_drivage = [17, 23, 28, 45, 60, 75, 101]
labels_drivage = ["18-23", "24-28", "29-45", "46-60", "61-75", "76+"]

df["DrivAgeBand"] = pd.cut(df["DrivAge"], bins=bornes_drivage, labels=labels_drivage)

# Verification : frequence et exposition par bande finale.
verif = (
    df.groupby("DrivAgeBand", observed=True)
      .apply(lambda g: pd.Series({
          "expo": g["Exposure"].sum(),
          "frequence": g["ClaimNb"].sum() / g["Exposure"].sum(),
          "n_polices": len(g),
          "part_expo": g["Exposure"].sum() / df["Exposure"].sum(),
      }), include_groups=False)
)
print(verif)

**Validation des bandes d'âge conducteur.**

Les six bandes sont saines : la plus petite (18-23) pèse 9 300 années-police
(3 % du portefeuille), suffisant pour une estimation fiable. Le contraste de
fréquence est préservé (0.20 jeune, 0.12 transition, plateau 0.09-0.10). Les
bandes du plateau (29-45, 46-60, 61-75, 76+) ont des fréquences proches ; on les
conserve distinctes car elles pourraient se différencier sur la **sévérité**
(Phase 2), et le GLM tranchera leur significativité statistique. Fusion éventuelle
sur preuve, pas sur intuition.

### 7.2 Bandes d'âge véhicule

Même approche que pour l'âge conducteur : vérifier la forme empirique avant de
découper. L'effet de l'âge du véhicule sur le risque est non linéaire et souvent
opposé entre fréquence et sévérité (un véhicule neuf : sécurité active mais
réparations coûteuses ; un vieux : l'inverse). Le découpage doit aussi neutraliser
les sentinelles à 99-100 ans (section 6.4) en les plaçant dans une bande haute.

In [ ]:
# --- VERIFICATION EMPIRIQUE : frequence par age vehicule ---
# Tranches fines jusqu'a 30 ans, puis un groupe "30+" qui isole les sentinelles.
bornes_test = list(range(0, 31, 2)) + [101]
tranches_veh = pd.cut(df["VehAge"], bins=bornes_test, right=False)
freq_veh = (
    df.groupby(tranches_veh, observed=True)
      .apply(lambda g: pd.Series({
          "expo": g["Exposure"].sum(),
          "frequence": g["ClaimNb"].sum() / g["Exposure"].sum(),
          "n_polices": len(g),
      }), include_groups=False)
)
print(freq_veh)

# Visualisation
fig, ax = plt.subplots(figsize=(12, 5))
freq_veh["frequence"].plot(kind="bar", ax=ax, color="darkgreen")
ax.axhline(df["ClaimNb"].sum() / df["Exposure"].sum(), color="red",
           linestyle="--", label="Frequence moyenne portefeuille")
ax.set_title("Frequence par age vehicule")
ax.set_ylabel("Frequence")
ax.set_xlabel("Age vehicule (tranches de 2 ans, dernier groupe = 30+)")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Lecture de la fréquence par âge véhicule : décroissance monotone.**

Motif net et distinct de l'âge conducteur : fréquence de **0.16 pour les véhicules
de moins de 2 ans**, puis chute immédiate vers un plateau à 0.09-0.10 jusqu'à ~12
ans, et décroissance régulière ensuite jusqu'à 0.04 pour les plus âgés. Plus le
véhicule est vieux, moins il sinistre.

Interprétation actuarielle (contre-intuitive mais robuste) : le surrisque des
véhicules neufs est **comportemental**, pas mécanique. Un véhicule neuf est
conduit plus intensivement et son propriétaire déclare le moindre sinistre
(valeur élevée, réparation prise en charge). Un vieux véhicule roule moins et son
propriétaire ne déclare pas les petits sinistres (franchise proche de sa valeur).

Cet effet porte sur la **fréquence** ; on attend l'effet inverse sur la
**sévérité** (un véhicule neuf coûte plus cher à réparer), ce qui justifie de
modéliser fréquence et sévérité séparément (Phase 2). Les sentinelles 99-100 ans
tombent dans le groupe 30+ (fréquence 0.04, cohérente) : neutralisées sans
déformation.

In [ ]:
# --- BANDES D'AGE VEHICULE ---
# Calees sur le signal : decroissance monotone. Bande fine "0-1" pour isoler le
# surrisque des vehicules neufs (0.16), plateau "2-11", puis regroupement des
# ages eleves en suivant la decroissance. La bande "20+" absorbe les sentinelles
# 99-100 (section 6.4) sans imputation.
bornes_vehage = [-1, 1, 5, 11, 15, 19, 101]
labels_vehage = ["0-1", "2-5", "6-11", "12-15", "16-19", "20+"]

df["VehAgeBand"] = pd.cut(df["VehAge"], bins=bornes_vehage, labels=labels_vehage)

# Verification : frequence et exposition par bande finale.
verif_veh = (
    df.groupby("VehAgeBand", observed=True)
      .apply(lambda g: pd.Series({
          "expo": g["Exposure"].sum(),
          "frequence": g["ClaimNb"].sum() / g["Exposure"].sum(),
          "n_polices": len(g),
          "part_expo": g["Exposure"].sum() / df["Exposure"].sum(),
      }), include_groups=False)
)
print(verif_veh)

**Validation des bandes d'âge véhicule.**

Les six bandes sont saines : la plus petite (20+) pèse 7 144 années-police (2 %),
suffisant pour l'estimation. Le gradient de fréquence est monotone décroissant et
lisible d'une bande à l'autre (0.16 neuf, plateau 0.09-0.10, puis 0.08, 0.07,
0.06). Contrairement à l'âge conducteur (plateau plat), chaque bande porte ici une
information distincte : découpage directement exploitable par le GLM. Sentinelles
99-100 absorbées dans 20+ (fréquence 0.06, cohérente).

### 7.3 Transformation de la densité

`Density` (densité de population de la commune) est une variable continue très
étalée : de 1 à 27 000 habitants/km², médiane à 393, distribution fortement
asymétrique à droite (quatre ordres de grandeur). En l'état, un GLM linéaire
donnerait un poids disproportionné aux rares zones ultra-denses. La transformation
$\log(\text{Density})$ comprime l'échelle et linéarise l'effet. On vérifie
d'abord la relation empirique entre densité et fréquence, en échelle log.

In [ ]:
# --- RELATION DENSITE / FREQUENCE EN ECHELLE LOG ---
# On decoupe la densite en deciles (10 groupes d'egale taille) et on regarde
# la frequence de chacun. Si la relation est reguliere en echelle log, cela
# confirme que log(Density) est la bonne transformation.
df["log_Density"] = np.log(df["Density"])

deciles = pd.qcut(df["Density"], q=10)
freq_density = (
    df.groupby(deciles, observed=True)
      .apply(lambda g: pd.Series({
          "density_min": g["Density"].min(),
          "density_max": g["Density"].max(),
          "frequence": g["ClaimNb"].sum() / g["Exposure"].sum(),
          "expo": g["Exposure"].sum(),
      }), include_groups=False)
)
print(freq_density)

# Visualisation : frequence en fonction de log(Density) moyen par decile.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution de la densite brute vs log.
sns.histplot(df["Density"], bins=50, ax=axes[0])
axes[0].set_title("Density (brute) : tres asymetrique")
axes[0].set_xlabel("Density (hab/km2)")

sns.histplot(df["log_Density"], bins=50, ax=axes[1], color="purple")
axes[1].set_title("log(Density) : distribution etalee")
axes[1].set_xlabel("log(Density)")

plt.tight_layout()
plt.show()

**Lecture de la densité et validation de la transformation log.**

*Distribution.* `Density` brute est inexploitable pour un modèle linéaire : 380 000
polices écrasées dans le premier bin, longue traîne vide, paquet isolé des zones
ultra-denses (jusqu'à 27 000 hab/km²). En échelle log, la distribution s'étale
proprement entre 2 et 10, exploitable par le GLM. Transformation $\log$ validée.

*Relation avec la fréquence.* Croissance monotone et régulière de la fréquence,
de 0.08 (< 33 hab/km²) à 0.13 (> 4 172 hab/km²), sur tous les déciles. Même signal
d'urbanisation que `Area` (section 3), mais en version continue et plus fine. La
régularité de la relation **en échelle log** confirme que $\log(\text{Density})$
est la forme fonctionnelle appropriée : l'effet sur le log du risque y est
approximativement linéaire.

### 7.4 Redondance entre variables : détection et traitement

Avant de construire le GLM, on scanne les corrélations entre variables numériques
pour repérer d'éventuelles redondances. Deux variables très corrélées portent la
même information : les inclure ensemble crée de la colinéarité (coefficients
instables, ininterprétables), rédhibitoire pour l'audit de la Phase 4.

Une corrélation forte est un **signal d'alerte, pas une explication** : elle
indique où inspecter, pas quoi décider. On lit d'abord la matrice pour repérer
les paires suspectes, puis on creuse chaque cas au niveau de la relation concrète
(bornes, distribution) pour comprendre la nature de la redondance avant de trancher.

`Area` étant catégorielle ordinale (A à F), on l'encode en rang (A=0 ... F=5) pour
l'inclure dans la matrice.

In [ ]:
# --- SCAN DES CORRELATIONS (detection des redondances) ---
# Un corrplot repere d'un coup d'oeil les variables redondantes. Area etant
# categorielle ordinale (A..F), on l'encode en rang pour l'inclure. Les
# correlations fortes signalent OU inspecter, pas quoi decider : on creuse
# ensuite au cas par cas (cf. Area/Density, bornes emboitees).
num_cols = ["ClaimNb", "Exposure", "VehPower", "VehAge", "DrivAge",
            "BonusMalus", "Density", "log_Density"]
corr_df = df[num_cols].copy()
corr_df["Area_rang"] = df["Area"].cat.codes  # A=0 .. F=5

corr = corr_df.corr(method="pearson")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Matrice de correlation (variables numeriques)")
plt.tight_layout()
plt.show()

**Lecture de la matrice de corrélation.**

*Redondance `Area` / `Density` (attendue).* `log_Density` et `Area_rang` à 0.97 :
confirmation de la redondance. `Area_rang` corrèle à 0.97 avec `log_Density` mais
seulement 0.59 avec `Density` brute, ce qui confirme que `Area` est un découpage
de l'échelle **log** de la densité. Décision : garder `log_Density` (voir 7.4).

*Relation `DrivAge` / `BonusMalus` (nouvelle, à -0.48).* Corrélation négative
modérée : les conducteurs âgés ont un bonus-malus plus bas (meilleur), ayant eu
le temps d'accumuler du bonus vers le coefficient plancher de 50, là où les jeunes
démarrent à 100. Âge et bonus-malus partagent l'information "expérience de
conduite", sans être redondants (|0.48| loin du seuil de colinéarité ~0.8) : le
bonus-malus capte le comportement individuel, l'âge capte des effets non liés à
l'expérience. **On garde les deux**, mais leurs coefficients seront à interpréter
conjointement, et la colinéarité surveillée via les VIF en Phase 2.

*Reste.* Aucune autre corrélation notable. `VehPower` décorrélé de tout. `ClaimNb`
sans corrélation forte avec un prédicteur isolé : le risque est multivarié, d'où
le besoin du modèle.

In [ ]:
# --- REDONDANCE AREA / DENSITY ---
# Si Area est une simple discretisation de Density, alors log_Density varie
# fortement d'une categorie Area a l'autre et peu a l'interieur. On le verifie.
redondance = (
    df.groupby("Area", observed=True)
      .apply(lambda g: pd.Series({
          "log_density_moyen": g["log_Density"].mean(),
          "log_density_min": g["log_Density"].min(),
          "log_density_max": g["log_Density"].max(),
          "density_median": g["Density"].median(),
          "n_polices": len(g),
      }), include_groups=False)
)
print(redondance)

# Correlation entre Area (encodee en rang ordinal A=0..F=5) et log_Density.
area_rang = df["Area"].cat.codes  # A=0, B=1, ..., F=5
correlation = np.corrcoef(area_rang, df["log_Density"])[0, 1]
print(f"\nCorrelation (Area ordinale, log_Density) : {correlation:.3f}")

**Décision : conserver `log_Density`, écarter `Area` du modèle.**

Les bornes min/max de `log_Density` par catégorie `Area` s'emboîtent exactement
(A : jusqu'à 3.91 ; B : 3.91 à 4.61 ; C : 4.61 à 6.21...) : `Area` n'est pas
seulement corrélée à `Density`, elle en est une **discrétisation directe** en 6
tranches. Corrélation de 0.971. Ce sont deux formes de la même information.

Les inclure ensemble créerait de la colinéarité (coefficients instables,
ininterprétables), rédhibitoire pour l'audit de la Phase 4. On conserve
**`log_Density`** : relation log-linéaire nette (rien perdu sur la forme),
parcimonie (1 coefficient contre 5), effet interprétable comme une élasticité.
`Area` gardée en réserve pour un test de robustesse (Phase 2).

Note (Phase 4) : `Density` est un **proxy géographique** (corrélé à la composition
socio-économique des territoires), à examiner sous l'angle de la non-discrimination.

### 7.5 Regroupement des modalités rares

`VehBrand` (11 marques) et `Region` (22 régions) sont catégorielles : le GLM
estime un coefficient par modalité. Une modalité à faible exposition donne un
coefficient instable (grande variance, signe parfois aberrant) et fragilise le
pipeline train/test si elle est absente d'un des jeux. On mesure l'exposition et
la sinistralité par modalité, puis on regroupe les plus rares dans une catégorie
"Autres". Principe constant : regrouper ce qui est trop maigre pour être estimé,
pas par principe.

In [ ]:
# --- EXPOSITION ET SINISTRALITE PAR MARQUE DE VEHICULE ---
def profil_modalite(colonne):
    return (
        df.groupby(colonne, observed=True)
          .apply(lambda g: pd.Series({
              "expo": g["Exposure"].sum(),
              "part_expo": g["Exposure"].sum() / df["Exposure"].sum(),
              "frequence": g["ClaimNb"].sum() / g["Exposure"].sum(),
              "n_polices": len(g),
          }), include_groups=False)
          .sort_values("expo", ascending=False)
    )

print("=== VehBrand (11 marques) ===")
print(profil_modalite("VehBrand"))
print("\n=== Region (22 regions) ===")
print(profil_modalite("Region"))

**Lecture : VehBrand conservée telle quelle, Region à regrouper.**

*VehBrand.* Répartition saine : 11 marques, toutes exploitables. La plus petite,
B14, pèse 1 % de l'exposition (2 270 années-police), à la limite basse mais
estimable. Comme c'est la seule marque marginale, la regrouper créerait une
catégorie "Autres" à une seule modalité, ce qui n'a pas de sens. **Aucun
regroupement** : on garde les 11 marques, le coefficient de B14 sera estimé avec
une incertitude un peu plus large, ce qui est acceptable.

*Region.* 22 modalités avec une longue traîne de régions minuscules : sous R94,
plusieurs régions pèsent moins de 0.5 % de l'exposition (R43 à 564 années-police,
R21 et R42 à ~1 200). Leurs fréquences sont erratiques (R94 à 0.14, R83 à 0.08) :
bruit d'échantillonnage, pas signal. Un coefficient GLM y serait instable. **On
regroupe** les régions sous 1 % de l'exposition dans une catégorie "Autres",
ramenant de 22 à 14 modalités.

In [ ]:
# --- REGROUPEMENT DES REGIONS RARES ---
# Regions sous 1 % de l'exposition : coefficient GLM non fiable (frequences
# erratiques par bruit d'echantillonnage). On les fusionne dans "Autres".
seuil = 0.01  # 1 % de l'exposition totale
expo_par_region = df.groupby("Region", observed=True)["Exposure"].sum()
part_region = expo_par_region / df["Exposure"].sum()

regions_gardees = part_region[part_region >= seuil].index.tolist()
regions_rares = part_region[part_region < seuil].index.tolist()

print(f"Regions gardees ({len(regions_gardees)}) : {sorted(regions_gardees)}")
print(f"Regions regroupees dans 'Autres' ({len(regions_rares)}) : {sorted(regions_rares)}")

# Nouvelle variable : Region regroupee.
df["RegionGrouped"] = df["Region"].astype(str).where(
    df["Region"].isin(regions_gardees), other="Autres"
)

# Verification de la nouvelle repartition.
verif_region = (
    df.groupby("RegionGrouped", observed=True)
      .apply(lambda g: pd.Series({
          "expo": g["Exposure"].sum(),
          "part_expo": g["Exposure"].sum() / df["Exposure"].sum(),
          "frequence": g["ClaimNb"].sum() / g["Exposure"].sum(),
      }), include_groups=False)
      .sort_values("expo", ascending=False)
)
print("\nRepartition apres regroupement :")
print(verif_region)

**Validation du regroupement des régions.**

La catégorie "Autres" agrège 8 régions pour 16 210 années-police (5 % de
l'exposition), avec une fréquence de 0.11, proche de la moyenne du portefeuille
(0.10). Exposition largement suffisante pour un coefficient stable ; "Autres" est
désormais plus grosse que 6 des régions conservées. Le regroupement remplace 8
coefficients non fiables par un seul, robuste. La fréquence légèrement au-dessus
de la moyenne (0.11) sera captée par un coefficient faiblement positif, sans perte
de signal.

## 7.6 Sauvegarde du jeu nettoyé et enrichi

On sauvegarde le portefeuille prêt à modéliser dans `data/processed/`, séparé de
la donnée brute (`data/raw/`, intouchée). La Phase 2 chargera ce fichier
directement. Il contient : les plafonnements (exposition à 1, ClaimNb à 4), les
features créées (`DrivAgeBand`, `VehAgeBand`, `log_Density`, `RegionGrouped`), et
`ClaimNbSev` pour la modélisation de la sévérité.

In [ ]:
# --- VERIFICATION : toutes les transformations sont-elles en place ? ---
# On controle que le df courant contient bien nettoyage + features avant d'ecrire.
attendues = ["DrivAgeBand", "VehAgeBand", "log_Density", "RegionGrouped", "ClaimNbSev"]
manquantes = [c for c in attendues if c not in df.columns]
assert not manquantes, f"Colonnes manquantes : {manquantes} (relance les cellules concernees)"

# Controle des plafonnements.
assert df["Exposure"].max() <= 1.0, "Exposition non plafonnee : relance section 6.2"
assert df["ClaimNb"].max() <= 4, "ClaimNb non ecrete : relance section 6.3"

print("Toutes les transformations sont en place.")
print(f"Colonnes du jeu final ({df.shape[1]}) : {list(df.columns)}")
print(f"Dimensions : {df.shape}")

In [ ]:
# --- SAUVEGARDE DU JEU PROCESSED ---
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DIR / "fremtpl2_clean.parquet"
df.to_parquet(output_path, index=False)

print(f"Jeu nettoye sauvegarde : {output_path}")
print(f"Dimensions : {df.shape[0]:,} polices, {df.shape[1]} colonnes")

## 8. Synthèse : 5 insights métier

Condensé de l'EDA. Chaque insight associe un fait chiffré à sa conséquence pour
la tarification.

### Insight 1 : l'exposition change tout, la fréquence de référence est 10 %

La fréquence du portefeuille est de **0.1007 sinistre par année-police** (10.1
pour 100), et non 0.053 (la moyenne naïve des comptages). L'écart d'un facteur 2
vient de l'exposition moyenne de 0.53 an : la moitié des contrats ne sont observés
qu'une partie de l'année. **Conséquence** : la fréquence se modélise comme un taux,
via un offset `log(Exposure)` dans le GLM Poisson. Ignorer l'exposition
sous-estimerait le risque de moitié.

### Insight 2 : le risque est très concentré, la sévérité a une queue extrême

Le **top 1 % des sinistres porte 37 % de la charge totale** (skewness de 116,
maximum à 4 M EUR pour un seul sinistre corporel). La distribution des coûts est
grossièrement log-normale. **Conséquence** : la sévérité se modélise avec une loi
Gamma ou log-normale (jamais gaussienne), et la question de l'écrêtement des
sinistres graves se pose. C'est le fondement de la réassurance.

### Insight 3 : le jeune conducteur est le facteur de risque n°1

La fréquence des **18-23 ans est de 0.20, le double de la moyenne**, puis chute
et se stabilise (distribution en L, pas en U : pas de surrisque des conducteurs
âgés dans ce portefeuille). **Conséquence** : l'âge est découpé en bandes fines en
bas (où l'information se concentre) et large sur le plateau. C'est le signal le
plus fort du portefeuille.

### Insight 4 : l'urbanisation structure le risque, et c'est un proxy sensible

La fréquence croît continûment de 0.08 (zones rurales) à 0.13 (zones denses).
`Area` s'est révélée être une **discrétisation directe de `log(Density)`**
(corrélation 0.97, bornes emboîtées) : on n'en garde qu'une pour éviter la
colinéarité. **Conséquence** : `log_Density` entre au modèle. Mais la densité est
un **proxy géographique** corrélé à la composition socio-économique des
territoires, à auditer sous l'angle de la non-discrimination (Phase 4).

### Insight 5 : les données portent des anomalies structurantes à documenter

freMTPL2 contient des anomalies qui changent la modélisation : **27 % des sinistres
déclarés n'ont aucun montant** (traités comme sans indemnisation, prime pure de
référence 167 EUR), des masses forfaitaires (atomes de probabilité), et des
valeurs sentinelles d'âge. **Conséquence** : fréquence et sévérité se modélisent
sur des comptages différents (`ClaimNb` vs `ClaimNbSev`), réconciliés par un
facteur 0.732. Savoir cela distingue une analyse rigoureuse d'un `.fit()` naïf.